# 71 The model's own answers, then the preference pairs (A100)

Method "cost first, then accuracy": (1) a cost-guided search finds the cheapest correct query known for every train question; (2) the current model (`rl_v1_it8`) answers every train question 8 times; (3) preference pairs: chosen = the cheap query, rejected = the model's own costlier or wrong answer; (4) DPO from the current model; (5) RL from the DPO model, with the cheapest known query as the cost yardstick; (6) the model is chosen on validation by a fixed rule, then dev and mini-dev are scored once.

Steps 2 and 3. The current model (`rl_v1_it8`) answers every train question 8 times, as it would write them (temperature 1), validation databases included; every answer is executed and labelled (correct with its cost, wrong, first-row pick, foreign constant). Nothing is trained.

Then the pairs, per question with at least one efficiency pair: up to 3 efficiency pairs (chosen = the cheap target, rejected = one of the model's own correct answers costing at least 10% and 50 steps more, the most frequent first) and 1 correctness pair (rejected = its most frequent wrong answer). Also the pairs of the three ablations.

**Runtime:** A100, about 1 to 1.5 hours (9 rounds of about 1,000 questions; the model is loaded once per round). **Resumable:** after a disconnect, run the setup and config cells (and the staging cell if there is one) again, then the cell that was running: finished work is kept.

**Then:** send me a message; I check the pair counts (decision D2) before DPO.

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
EXPORT = "sft_v1"                   # SFT export: its 4 held-out train databases are the validation set
CURRENT = "rl_v1_it8"               # our current model (RL, kept on validation): the start and the reference
SEARCH = "search_v1"                # 1. cost-guided search, all train questions -> cheap targets
OWN = "own_v1"                      # 2. the current model's own answers, 8 per train question
PAIRS = "pairs_v2"                  # 3. preference pairs (main)
DPO = "dpo_v2"                      # 4. DPO from CURRENT, 2 epochs: dpo_v2_e1 (after epoch 1) and dpo_v2
DPO_RUNS = [f"{DPO}_e1", DPO]
RL = "rl_v2"                        # 5. RL from the DPO model kept on validation: rl_v2_it8 and rl_v2
RL_RUNS = [f"{RL}_it8", RL]
# Ablations (DPO only, same settings as the kept DPO model): where the chosen and rejected answers come from.
ABLATIONS = {"gold": "--chosen gold",      # chosen = gold: the generated queries are not used at all
             "pool": "--chosen pool",      # chosen = the first pool's cheapest query: no search
             "off": "--rejected pool"}     # rejected = other models' pool queries, not the model's own
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
VAL_DBS = ",".join(json.load(open(f"{R}/{EXPORT}/export_meta.json"))["holdout_db_ids"])
def done(run):
    return os.path.exists(f"{R}/{run}/adapter_meta.json")
print("validation databases:", VAL_DBS)
assert os.path.exists(f"{R}/{SEARCH}/cheap_targets_train.jsonl"), "run 70_search_full first"
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

### Stage the train databases (30 GB, a few minutes; skipped when already staged)

In [ ]:
!effsql stage --split train
!effsql schema-cache --split train

### 1. Own answers: 8 per train question from the current model (sampling only)

In [ ]:
!python -m effsql.train rl --run {OWN} --init-from {CURRENT} --settings own_answers

### 2. Pairs (CPU work, a few minutes): main, then the ablations

In [ ]:
!python -m effsql.train pairs --targets-run {SEARCH} --own-run {OWN} --run {PAIRS}
for name, opt in ABLATIONS.items():
    !python -m effsql.train pairs --targets-run {SEARCH} --own-run {OWN} --run {PAIRS}_{name} {opt}

### 3. Summary

`questions`: per question, whether it got pairs, or why not (`writes_cheap_form`: every correct answer of the model is already within 10% of the target; `only_wrong`; `no_answers`). `chosen_source`: where the chosen queries come from (gold, first pool, search).

In [ ]:
for run in [PAIRS] + [f"{PAIRS}_{n}" for n in ABLATIONS]:
    m = json.load(open(f"{R}/{run}/export_meta.json"))
    for part in ("train", "val"):
        s = m["stats"][f"dpo_{part}"]
        print(f"{run:<16}{part:<6}{s['n_questions']:>6} questions {s['n_examples']:>6} pairs ", s["by_type"], s["questions"], s["chosen_source"])
    print()
s = json.load(open(f"{R}/{OWN}/rl_summary.json"))
n = sum(i["n_questions"] for i in s["iterations"])
print(f"own answers: {len(s['iterations'])} rounds, {n} questions; correct %:", [round(i["score_pct_correct"], 1) for i in s["iterations"]])

### Done: disconnect the runtime